In [ ]:
import os
import sqlite3
import numpy as np
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
import pandas as pd

# LangChain 관련
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain.schema import Document
from langchain.docstore.in_memory import InMemoryDocstore
from langchain.prompts import PromptTemplate
import faiss

# ----------------------------
# 환경설정
# ----------------------------
load_dotenv("env.txt")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ----------------------------
# 관계형 DB 설정 (SQLite)
# ----------------------------
DB_PATH = "user_history.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS user_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    starttime TEXT NOT NULL,
    text TEXT NOT NULL,
    sentiment TEXT NOT NULL
)
""")
conn.commit()

# ----------------------------
# 감정별 프롬프트
# ----------------------------
sentiment_prompts = {
    "Angry": "사용자가 화가 난 상태입니다. 화난 말투 표현.",
    "Happy": "사용자가 행복한 상태입니다. 행복한 말투 표현.",
    "Sad": "사용자가 슬픈 상태입니다. 슬픈 말투 표현.",
    "Disgust": "사용자가 역겨움/불쾌함을 표현했습니다. 역겨움, 불쾌함 공감.",
    "Neutral": "사용자가 중립적입니다. 일반적인 정보 제공과 자연스러운 대화를 이어가세요.",
    "Surprise": "사용자가 놀람을 표현했습니다. 놀람의 이유를 묻거나 공감하며 대화를 이어가세요.",
    "Fear": "사용자가 두려움을 표현했습니다. 안정감을 주고 안전한 느낌을 전달하세요."
}

# ----------------------------
# 전역 상태 (유저별)
# ----------------------------
user_states: Dict[str, Dict[str, Any]] = {}

VEC_PATH = "vector_store_faiss"

# ----------------------------
# LangChain Embeddings & FAISS
# ----------------------------
embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
dim = len(embeddings_model.embed_query("test"))

index = faiss.IndexFlatL2(dim)
vecdb = FAISS(
    index=index,
    embedding_function=embeddings_model,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

# ----------------------------
# 관계형 DB → VectorDB 동기화
# ----------------------------
def add_to_vecdb(rows: List[Dict[str, Any]]):
    if not rows:
        return
    texts = [f"User: {r['username']}\nText: {r['text']}\nSentiment: {r['sentiment']}" for r in rows]
    docs = [Document(page_content=t, metadata=r) for t, r in zip(texts, rows)]
    vecdb.add_documents(docs)
    vecdb.save_local(VEC_PATH)

def sync_db_to_vecdb(username: Optional[str] = None):
    query = "SELECT username, starttime, text, sentiment FROM user_history"
    if username:
        query += f" WHERE username='{username}'"
    df = pd.read_sql(query, conn)
    if df.empty:
        return
    add_to_vecdb(df.to_dict(orient="records"))

# ----------------------------
# GPT 모델 & PromptTemplate
# ----------------------------
chat_model = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini",
    temperature=0.3
)

prompt_template = PromptTemplate(
    input_variables=["final_prompt"],
    template="""너는 'AI 인형'이라는 가상 캐릭터야.
유저가 제공한 대화 내용과 요약 정보(final_prompt)에 따라 행동해야 해.
행동 지침:
1. 유저의 감정과 상황을 파악하고 공감해줘.
2. 유저의 최근 대화, 중요한 문장, 장기 기억 요약을 참고하여 맥락에 맞게 답변.
3. 최근 대화로 지금 말하고 있는 맥락 파악.
4. 중요한 문장은 유저에 대한 특성 파악.
5. 장기 기억 요약으로 유저에 대한 특별한 상황 인식.
6. 질문, 제안, 조언 등을 적절히 섞어 자연스럽게 대화 이어가기.
7. 1~3문장 정도로 간결하게 작성.

{final_prompt}"""
)

llm_pipeline = prompt_template | chat_model

# ----------------------------
# 요약 함수
# ----------------------------
def summarize_with_gpt(prompt: str) -> str:
    return llm_pipeline.invoke({"final_prompt": prompt})

def summarize_recent_context_str(text: str) -> str:
    prompt = f"다음 대화 내용을 주요 내용 중심으로 5문장 이하로 요약하세요:\n{text}"
    return summarize_with_gpt(prompt)

def summarize_rag_context_text(rag_text: str) -> str:
    if not rag_text.strip():
        return ""
    combined_text = " ".join(dict.fromkeys([line.strip() for line in rag_text.split("\n") if line.strip()]))
    prompt = f"다음 내용을 주요 내용 중심으로 5문장 이하로 요약하세요:\n{combined_text}"
    return summarize_with_gpt(prompt)

# ----------------------------
# AI 응답 처리 (유저별 상태)
# ----------------------------
def handle_user_input(username: str, starttime: str, text: str, sentiment: str) -> str:
    # 유저 상태 초기화
    if username not in user_states:
        user_states[username] = {
            "history": [],              # 모든 문장 저장
            "trigger_index": None,      # 트리거 인덱스
            "important_sentences_rows": []
        }
    state = user_states[username]

    # ------------------
    # 새 입력을 히스토리에 추가 (DB에는 바로 안넣음)
    # ------------------
    entry = {"starttime": starttime, "text": text, "sentiment": sentiment, "username": username}
    state["history"].append(entry)

    # ------------------
    # 트리거 조건: 감정 변화 or 특정 키워드
    # ------------------
    trigger = False
    if len(state["history"]) >= 2 and state["history"][-2]["sentiment"] != sentiment:
        trigger = True
    if any(kw in text for kw in ["따돌리는거 같아", "화가 나"]):  # 키워드 트리거 예시
        trigger = True

    if trigger:
        state["trigger_index"] = len(state["history"]) - 1

    # ------------------
    # 트리거 후 윈도우 완성 검사
    # ------------------
    completed = None
    if state["trigger_index"] is not None:
        trig_idx = state["trigger_index"]
        if len(state["history"]) >= trig_idx + 3:  # 트리거 + 이후 2문장 확보됨
            start_idx = max(0, trig_idx - 2)
            end_idx = trig_idx + 3
            selected = state["history"][start_idx:end_idx]

            combined_text = " ".join([h["text"] for h in selected])
            combined_row = {
                "username": username,
                "starttime": selected[0]["starttime"],
                "text": combined_text,
                "sentiment": state["history"][trig_idx]["sentiment"]
            }

            if not any(row["text"] == combined_text for row in state["important_sentences_rows"]):
                state["important_sentences_rows"].append(combined_row)

                # 이제 DB + VecDB에 저장
                cursor.execute(
                    "INSERT INTO user_history (username, starttime, text, sentiment) VALUES (?, ?, ?, ?)",
                    (combined_row["username"], combined_row["starttime"], combined_row["text"], combined_row["sentiment"])
                )
                conn.commit()
                add_to_vecdb([combined_row])

            state["trigger_index"] = None  # 초기화
            completed = combined_row

    # ------------------
    # RAG / LLM 응답 (이 부분은 기존과 동일)
    # ------------------
    query_embedding = np.array(embeddings_model.embed_query(text), dtype=np.float32)
    D, I = vecdb.index.search(query_embedding.reshape(1, -1), k=5)
    retrieved_docs = [vecdb.docstore._dict[vecdb.index_to_docstore_id[i]] for i in I[0] if i in vecdb.index_to_docstore_id]
    rag_context_text = "\n".join([doc.page_content for doc in retrieved_docs])
    rag_context = summarize_rag_context_text(rag_context_text)

    recent_conversation = state["history"][-5:] if len(state["history"]) > 5 else state["history"]
    conversation_text = "\n".join([f"User: {h['text']}" for h in recent_conversation])
    conversation_text = summarize_recent_context_str(conversation_text)

    current_prompt = sentiment_prompts.get(sentiment, sentiment_prompts["Neutral"])
    final_prompt_parts = [current_prompt]
    if rag_context:
        final_prompt_parts.append(f"참고 컨텍스트(RAG):\n{rag_context}")
    final_prompt_parts.append(f"현재 대화:\n{conversation_text}")
    final_prompt_parts.append("\nAI 인형 응답:")
    final_prompt = "\n".join(final_prompt_parts)

    response = llm_pipeline.invoke({"final_prompt": final_prompt}).content
    return response



In [2]:
# -----------------
# 멀티 유저 테스트 시나리오
# -----------------
test_inputs_multiuser_mixed = [
    # 1️⃣ 유저1: 학교/친구
    {"username":"user1", "starttime":"2025-08-30T16:30:00","text":"오늘도 점심을 혼자 먹었어","sentiment":"Sad"},
    {"username":"user2", "starttime":"2025-08-30T16:30:05","text":"시험이 다가와서 긴장돼","sentiment":"Fear"},
    {"username":"user3", "starttime":"2025-08-30T16:30:10","text":"새 프로젝트 때문에 걱정돼","sentiment":"Sad"},
    {"username":"user1", "starttime":"2025-08-30T16:30:15","text":"친구들이 점점 멀어지는 것 같아","sentiment":"Sad"},
    {"username":"user2", "starttime":"2025-08-30T16:30:20","text":"오늘 계획대로 공부를 거의 못했어","sentiment":"Sad"},
    {"username":"user3", "starttime":"2025-08-30T16:30:25","text":"하지만 조금씩 아이디어가 떠올라","sentiment":"Neutral"},

    # 2️⃣ 감정 전환 텀
    {"username":"user1", "starttime":"2025-08-30T16:30:30","text":"그래도 조금씩 먼저 인사하려고 해","sentiment":"Happy"},
    {"username":"user2", "starttime":"2025-08-30T16:30:35","text":"엄마가 조금 이해해주셔서 마음이 놓여","sentiment":"Happy"},
    {"username":"user3", "starttime":"2025-08-30T16:30:40","text":"오늘 드로잉 조금 해봤는데 재밌었어","sentiment":"Happy"},

    # 3️⃣ 유저4: 사회적 활동
    {"username":"user4", "starttime":"2025-08-30T16:30:45","text":"모임에서 긴장돼서 거의 말 못했어","sentiment":"Fear"},
    {"username":"user4", "starttime":"2025-08-30T16:30:50","text":"내가 소외되는 느낌이 들어","sentiment":"Sad"},
    {"username":"user1", "starttime":"2025-08-30T16:30:55","text":"서로 서먹하지만 조금씩 나아지고 있어","sentiment":"Neutral"},
    {"username":"user2", "starttime":"2025-08-30T16:31:00","text":"조금씩 집중하려고 노력 중이야","sentiment":"Neutral"},
    {"username":"user3", "starttime":"2025-08-30T16:31:05","text":"내일은 더 연습해서 발전해야지","sentiment":"Neutral"},

    # 4️⃣ 유저4 긍정 전환
    {"username":"user4", "starttime":"2025-08-30T16:31:10","text":"다음엔 먼저 말을 걸어볼까 생각 중이야","sentiment":"Neutral"},
    {"username":"user4", "starttime":"2025-08-30T16:31:15","text":"조금씩 적응하면 괜찮아질 거야","sentiment":"Happy"},
    {"username":"user1", "starttime":"2025-08-30T16:31:20","text":"오늘 용기내서 질문도 해봤어","sentiment":"Happy"},
    {"username":"user2", "starttime":"2025-08-30T16:31:25","text":"조금 위로받아서 기분이 나아졌어","sentiment":"Happy"},
    {"username":"user3", "starttime":"2025-08-30T16:31:30","text":"오늘 느낀 것들을 기록해두면 도움이 될 거야","sentiment":"Neutral"},

    # 5️⃣ 마무리 감정 정리
    {"username":"user4", "starttime":"2025-08-30T16:31:35","text":"오늘 경험을 기록해두면 도움이 될 거야","sentiment":"Neutral"},
    {"username":"user1", "starttime":"2025-08-30T16:31:40","text":"점점 친구들과 친해지고 있는 느낌이야","sentiment":"Happy"},
    {"username":"user2", "starttime":"2025-08-30T16:31:45","text":"계획표대로는 아니지만 조금씩 나아지고 있어","sentiment":"Neutral"},
    {"username":"user3", "starttime":"2025-08-30T16:31:50","text":"오늘은 다행히 성취감을 느꼈어","sentiment":"Happy"},
]


In [ ]:
# # -----------------
# # 멀티 유저 테스트 시나리오 (일주일 뒤)
# # -----------------
# test_inputs_multiuser_week_later = [
#     # 1️⃣ 유저1: 학교/친구
#     {"username":"user1", "starttime":"2025-09-06T16:30:00","text":"오늘은 점심시간에 혼자 있진 않았어","sentiment":"Neutral"},
#     {"username":"user2", "starttime":"2025-09-06T16:30:05","text":"시험이 다가오는데 지난주보다 마음이 안정돼","sentiment":"Neutral"},
#     {"username":"user3", "starttime":"2025-09-06T16:30:10","text":"프로젝트가 조금 진전돼서 기분이 나아","sentiment":"Happy"},
#     {"username":"user1", "starttime":"2025-09-06T16:30:15","text":"친구들이 먼저 말을 걸어줘서 좋았어","sentiment":"Happy"},
#     {"username":"user2", "starttime":"2025-09-06T16:30:20","text":"공부 계획을 조금 더 잘 지키게 되었어","sentiment":"Happy"},
#     {"username":"user3", "starttime":"2025-09-06T16:30:25","text":"새로운 아이디어도 떠올랐어","sentiment":"Happy"},

#     # 2️⃣ 유저4: 사회적 활동/심리적 상태
#     {"username":"user4", "starttime":"2025-09-06T16:30:30","text":"모임에서 긴장하긴 했지만 전보다는 낫더라","sentiment":"Neutral"},
#     {"username":"user4", "starttime":"2025-09-06T16:30:35","text":"몇 명과는 친해진 느낌이야","sentiment":"Happy"},
#     {"username":"user1", "starttime":"2025-09-06T16:30:40","text":"오늘은 용기내서 발표도 했어","sentiment":"Happy"},
#     {"username":"user2", "starttime":"2025-09-06T16:30:45","text":"집중력이 조금 더 생긴 것 같아","sentiment":"Neutral"},
#     {"username":"user3", "starttime":"2025-09-06T16:30:50","text":"내 드로잉 실력이 조금 나아진 느낌","sentiment":"Happy"},

#     # 3️⃣ 유저1/유저4 감정 혼합
#     {"username":"user1", "starttime":"2025-09-06T16:30:55","text":"친구들과 농담도 주고받았어","sentiment":"Happy"},
#     {"username":"user4", "starttime":"2025-09-06T16:31:00","text":"조금 긴장됐지만 대화가 즐거웠어","sentiment":"Happy"},
#     {"username":"user2", "starttime":"2025-09-06T16:31:05","text":"조금 지쳤지만 성취감이 있어","sentiment":"Neutral"},
#     {"username":"user3", "starttime":"2025-09-06T16:31:10","text":"계획했던 것들을 하나씩 완료했어","sentiment":"Happy"},

#     # 4️⃣ 유저4 긍정 강화
#     {"username":"user4", "starttime":"2025-09-06T16:31:15","text":"오늘 경험을 기록해두면 다음에도 도움이 될 거야","sentiment":"Neutral"},
#     {"username":"user1", "starttime":"2025-09-06T16:31:20","text":"오늘 친구들과 관계가 조금 더 가까워진 느낌","sentiment":"Happy"},
#     {"username":"user2", "starttime":"2025-09-06T16:31:25","text":"집중력이 생기니 공부가 더 수월했어","sentiment":"Happy"},
#     {"username":"user3", "starttime":"2025-09-06T16:31:30","text":"드로잉 결과물이 마음에 들어","sentiment":"Happy"},
#     {"username":"user4", "starttime":"2025-09-06T16:31:35","text":"다음 모임이 기대돼","sentiment":"Happy"},

#     # 5️⃣ 마무리
#     {"username":"user1", "starttime":"2025-09-06T16:31:40","text":"오늘 하루가 즐거웠어","sentiment":"Happy"},
#     {"username":"user2", "starttime":"2025-09-06T16:31:45","text":"계획대로는 아니지만 조금씩 성장하고 있어","sentiment":"Neutral"},
#     {"username":"user3", "starttime":"2025-09-06T16:31:50","text":"오늘 하루 만족스러워","sentiment":"Happy"},
#     {"username":"user4", "starttime":"2025-09-06T16:31:55","text":"오늘 하루 경험을 기록해두니 마음이 안정돼","sentiment":"Neutral"},
# ]


In [3]:
# 반복문으로 테스트
for input_data in test_inputs_multiuser_mixed:
    response = handle_user_input(
        username=input_data["username"],
        starttime=input_data["starttime"],
        text=input_data["text"],
        sentiment=input_data["sentiment"]
    )
    print(f"--- User (유저 이름: {input_data['username']}) (감정: {input_data['sentiment']}) ---")
    print(input_data["text"])
    print(f"--- AI Response ---")
    print(response)
    print("\n")

--- User (유저 이름: user1) (감정: Sad) ---
오늘도 점심을 혼자 먹었어
--- AI Response ---
혼자 점심을 먹는 것이 외롭고 힘들게 느껴진다니, 정말 안타깝네요. 소속감이나 연결이 부족하다는 느낌이 드는 것은 자연스러운 일이에요. 다음에는 친구나 동료에게 함께 식사하자고 제안해보는 것이 어떨까요? 작은 변화가 기분에 긍정적인 영향을 줄 수 있을 거예요. 이런 제안에 대해 어떻게 생각하시나요?


--- User (유저 이름: user2) (감정: Fear) ---
시험이 다가와서 긴장돼
--- AI Response ---
시험이 다가오면서 느끼는 긴장감은 정말 자연스러운 감정이에요. 지금 1분 동안 깊게 숨을 쉬어보는 것이 도움이 될 수 있어요. 숨을 들이쉬고 내쉬는 것에 집중해 보세요. 어떤 방법이 가장 도움이 될 것 같나요?


--- User (유저 이름: user3) (감정: Sad) ---
새 프로젝트 때문에 걱정돼
--- AI Response ---
새 프로젝트로 인해 걱정이 많으시군요. 그런 감정을 느끼는 것은 정말 자연스러운 일이에요. "이 프로젝트가 잘 진행될 수 있는 방법이 있을까?"라는 긍정적인 질문으로 생각을 전환해 보세요. 오늘은 프로젝트에 대한 긍정적인 면을 적어보는 작은 행동을 시도해 보시는 건 어떨까요? 어떤 부분이 가장 걱정되시나요?


--- User (유저 이름: user1) (감정: Sad) ---
친구들이 점점 멀어지는 것 같아
--- AI Response ---
혼자 점심을 먹는 것은 정말 외롭고 힘든 경험이죠. 친구들과의 거리감이 느껴지는 것에 안타까움을 느끼는 당신의 마음을 이해해요. 친구들과의 관계를 회복하기 위해 어떤 작은 행동을 시도해볼 수 있을까요? 예를 들어, 먼저 연락해보는 건 어떨까요?


--- User (유저 이름: user2) (감정: Sad) ---
오늘 계획대로 공부를 거의 못했어
--- AI Response ---
### 감정 반영
시험이 다가오면서 긴장감과 

KeyboardInterrupt: 

In [15]:
# history 출력 예시
for i, entry in enumerate(history):
    print(f"--- Entry {i+1} ---")
    print(f"Starttime      : {entry['starttime']}")
    print(f"Text           : {entry['text']}")
    print(f"Sentiment      : {entry['sentiment']}")

    print("---------------------------\n")


--- Entry 1 ---
Starttime      : 2025-08-30T16:30:00
Text           : 오늘도 혼자 점심을 먹었어
Sentiment      : Sad
---------------------------

--- Entry 2 ---
Starttime      : 2025-08-30T16:30:10
Text           : 친구들이 점점 멀어지는 것 같아
Sentiment      : Sad
---------------------------

--- Entry 3 ---
Starttime      : 2025-08-30T16:30:20
Text           : 하지만 조금씩 먼저 인사하려고 해
Sentiment      : Happy
---------------------------

--- Entry 4 ---
Starttime      : 2025-08-30T16:30:30
Text           : 서로 서먹하지만 나아지고 있는 느낌
Sentiment      : Neutral
---------------------------

--- Entry 5 ---
Starttime      : 2025-08-30T16:30:40
Text           : 오늘은 용기내서 질문도 해봤어
Sentiment      : Happy
---------------------------

--- Entry 6 ---
Starttime      : 2025-08-30T16:31:00
Text           : 시험이 다가와서 스트레스가 심해
Sentiment      : Fear
---------------------------

--- Entry 7 ---
Starttime      : 2025-08-30T16:31:10
Text           : 오늘도 계획표대로 공부 못했어
Sentiment      : Sad
---------------------------

--- Entry 8 ---
Starttime  

In [20]:
print(response.content)

혼자 지내는 시간이 많아서 힘들겠어요. 그래도 용기 내서 인사한 건 정말 대단한 걸요! 성적에 대한 걱정도 여전히 크신 것 같고, 그런 마음 이해해요. 어떤 부분이 특히 힘든지 이야기해보면 좋을 것 같아요. 함께 고민해보면 조금이나마 도움이 될 수 있을 거예요.


# SQLite 3 버전으로 DB 기반 멀티 유저 맞춤 답변 출력 플로우 완료 (Relative DB 로 해당 유저 검색 + Vector DB 로 맞춤 답변 생성)